In [1]:
!pip install pandas sentence-transformers scikit-learn transformers torch

In [2]:
# %%
import os
import re
import json
from typing import List, Dict, Tuple
import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer
from sklearn.neighbors import NearestNeighbors

from transformers import T5ForConditionalGeneration, AutoTokenizer, pipeline
import torch

# reproducible device selection
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)


Using device: cpu


In [3]:
# %%
def load_or_example(faqs_path="faqs.csv", tickets_path="tickets.csv"):
    if os.path.exists(faqs_path):
        faqs = pd.read_csv(faqs_path)
    else:
        faqs = pd.DataFrame([
            {"id": 1, "question": "How do I apply for leave?", "answer": "Use the HR portal -> Leaves -> Apply. Provide dates and reason. Contact your supervisor for approval."},
            {"id": 2, "question": "How to request a laptop?", "answer": "Fill in the IT equipment request form in the intranet. Your mentor must approve the request."},
            {"id": 3, "question": "Where is the onboarding material?", "answer": "Onboarding slides and videos are available at the shared drive: /Company/Onboarding/Interns."},
            {"id": 4, "question": "How to access company VPN?", "answer": "Install the VPN client, register your device in the IT portal, then use your company credentials."},
        ])
        print("faqs.csv not found — using example FAQ data.")

    if os.path.exists(tickets_path):
        tickets = pd.read_csv(tickets_path)
    else:
        tickets = pd.DataFrame([
            {"id": 101, "subject": "VPN not connecting", "description": "I cannot connect to VPN from home. Error: TLS handshake failed.", "resolution": "Reinstalled client, cleared certificates and whitelisted device in IT portal."},
            {"id": 102, "subject": "Laptop battery issue", "description": "Battery drains too quick on my assigned laptop.", "resolution": "Submitted device for replacement; interim power plan adjusted."},
            {"id": 103, "subject": "Access to S3 bucket", "description": "Need read access to project s3 bucket to fetch datasets.", "resolution": "Added IAM role and provided bucket URL with temporary credentials."},
        ])
        print("tickets.csv not found — using example tickets data.")

    # Ensure cols exist
    for c in ["id", "question", "answer"]:
        if c not in faqs.columns:
            faqs[c] = ""
    for c in ["id", "subject", "description", "resolution"]:
        if c not in tickets.columns:
            tickets[c] = ""
    return faqs, tickets

faqs_df, tickets_df = load_or_example()
print("FAQ count:", len(faqs_df), "Tickets count:", len(tickets_df))

faqs.csv not found — using example FAQ data.
tickets.csv not found — using example tickets data.
FAQ count: 4 Tickets count: 3


In [4]:
# %%
def build_documents(faqs: pd.DataFrame, tickets: pd.DataFrame) -> List[Dict]:
    docs = []
    # FAQ entries: keep Q and A together
    for _, r in faqs.iterrows():
        text = f"Q: {r['question'].strip()} \nA: {r['answer'].strip()}"
        docs.append({"id": f"faq_{int(r['id'])}", "text": text, "source": "faq", "meta": {"question": r['question'], "answer": r['answer']}})
    # Tickets: subject + description + (optional) resolution
    for _, r in tickets.iterrows():
        text = f"Subject: {r['subject'].strip()} \nDescription: {r['description'].strip()} \nResolution: {r['resolution'].strip()}"
        docs.append({"id": f"ticket_{int(r['id'])}", "text": text, "source": "ticket", "meta": {"subject": r['subject'], "resolution": r['resolution']}})
    return docs

documents = build_documents(faqs_df, tickets_df)
print("Built documents:", len(documents))
# show one
documents[:2]

Built documents: 7


[{'id': 'faq_1',
  'text': 'Q: How do I apply for leave? \nA: Use the HR portal -> Leaves -> Apply. Provide dates and reason. Contact your supervisor for approval.',
  'source': 'faq',
  'meta': {'question': 'How do I apply for leave?',
   'answer': 'Use the HR portal -> Leaves -> Apply. Provide dates and reason. Contact your supervisor for approval.'}},
 {'id': 'faq_2',
  'text': 'Q: How to request a laptop? \nA: Fill in the IT equipment request form in the intranet. Your mentor must approve the request.',
  'source': 'faq',
  'meta': {'question': 'How to request a laptop?',
   'answer': 'Fill in the IT equipment request form in the intranet. Your mentor must approve the request.'}}]

In [5]:
# %%
EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
print("Loading embedding model:", EMBED_MODEL_NAME)
embedder = SentenceTransformer(EMBED_MODEL_NAME, device=DEVICE)

# Create list of texts
doc_texts = [d['text'] for d in documents]
doc_ids   = [d['id'] for d in documents]

# Compute embeddings (batch)
doc_embeddings = embedder.encode(doc_texts, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True)
print("Embeddings shape:", doc_embeddings.shape)

# Build NearestNeighbors (cosine via metric='cosine')
nn = NearestNeighbors(n_neighbors=5, metric="cosine").fit(doc_embeddings)

Loading embedding model: sentence-transformers/all-MiniLM-L6-v2


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings shape: (7, 384)


In [6]:
# %%
USE_GENERATOR = True  # set False to skip generator and return retrieval snippets
GEN_MODEL_NAME = "google/flan-t5-small"  # small, runs quickly; switch to larger if you want better quality

generator = None
tokenizer = None
if USE_GENERATOR:
    print("Loading generator:", GEN_MODEL_NAME)
    tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL_NAME)
    gen_model = T5ForConditionalGeneration.from_pretrained(GEN_MODEL_NAME).to(DEVICE)
    # Create a simple pipeline wrapper
    generator = pipeline("text2text-generation", model=gen_model, tokenizer=tokenizer, device=0 if DEVICE=="cuda" else -1, max_length=256)
    print("Generator loaded.")

Loading generator: google/flan-t5-small


tokenizer_config.json: 0.00B [00:00, ?B/s]

C:\Users\talha\Anaconda3new\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\talha\.cache\huggingface\hub\models--google--flan-t5-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Device set to use cpu


Generator loaded.


In [7]:
# %%
def retrieve(query: str, top_k: int = 3) -> List[Tuple[Dict,float]]:
    """
    Return top_k documents and similarity scores for query.
    """
    q_emb = embedder.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    distances, indices = nn.kneighbors(q_emb, n_neighbors=min(top_k, len(documents)))
    distances = distances[0]
    indices = indices[0]
    results = []
    for dist, idx in zip(distances, indices):
        score = 1 - dist  # approximate similarity since metric=cosine returns distance
        results.append((documents[idx], float(score)))
    return results

def build_prompt(query: str, retrieved: List[Tuple[Dict,float]]) -> str:
    """
    Build a prompt for the generator: include question and top retrieved contexts.
    Keep the prompt concise to avoid hitting model length limits.
    """
    prompt_parts = []
    prompt_parts.append("You are an assistant for interns at a company. Answer the intern's question concisely, politely, and with actionable steps. If the answer is not present in the context, provide a helpful generic answer and suggest resources or whom to contact.")
    prompt_parts.append("\n---\nContext (relevant documents):\n")
    for i, (doc, score) in enumerate(retrieved):
        # limit context size
        ctx = doc["text"]
        # optionally trim long contexts
        if len(ctx) > 800:
            ctx = ctx[:800] + " ..."
        prompt_parts.append(f"[Doc {i+1} | source={doc['source']} | score={score:.3f}]\n{ctx}\n")
    prompt_parts.append("\nQuestion:\n" + query + "\n")
    prompt_parts.append("\nAnswer:")
    return "\n".join(prompt_parts)

def generate_answer(query: str, top_k: int = 3, use_generator: bool = True) -> Dict:
    # Retriever
    retrieved = retrieve(query, top_k=top_k)
    # If generator is disabled or not loaded, return retrieval snippets
    if (not use_generator) or (generator is None):
        snippets = [ {"doc": r[0], "score": r[1]} for r in retrieved ]
        combined = "\n\n".join([f"(score={r[1]:.3f}) {r[0]['text']}" for r in retrieved])
        return {"query": query, "answer": combined, "type": "retrieval", "snippets": snippets}

    # Build prompt for generator
    prompt = build_prompt(query, retrieved)
    # Generate
    gen_out = generator(prompt, max_length=256, do_sample=False)
    answer_text = gen_out[0]['generated_text'].strip()
    # If generator output is empty or meaningless, fallback to retrieval snippets
    if len(answer_text) < 5:
        combined = "\n\n".join([f"(score={r[1]:.3f}) {r[0]['text']}" for r in retrieved])
        return {"query": query, "answer": combined, "type": "retrieval_fallback", "snippets": [ {"doc": r[0], "score": r[1]} for r in retrieved ]}
    return {"query": query, "answer": answer_text, "type": "generated", "snippets": [ {"doc": r[0], "score": r[1]} for r in retrieved ]}

In [10]:
# %%
examples = [
    "How can I apply for leave next week?",
    "My VPN is not connecting. What should I do?",
    "How do I request a laptop?",
    "Where can I find onboarding materials?",
    "How do I get access to the project data stored in S3?"
]

for q in examples:
    print("="*80)
    print("Q:", q)
    ans = generate_answer(q, top_k=3, use_generator=USE_GENERATOR)
    print("Type:", ans["type"])
    print("A:", ans["answer"])
    print()

Q: How can I apply for leave next week?
Type: generated
A: Use the HR portal -> Leaves -> Apply. Provide dates and reason. Contact your supervisor for approval. [Doc 1 | source=faq | score=0.706]

Q: My VPN is not connecting. What should I do?
Type: generated
A: [Doc 1 | source=ticket | score=0.630]

Q: How do I request a laptop?
Type: generated
A: Fill in the IT equipment request form in the intranet. Your mentor must approve the request. [Doc 2 | source=ticket | score=0.806]

Q: Where can I find onboarding materials?
Type: generated
A: Onboarding slides and videos are available at the shared drive: /Company/Onboarding/Interns. [Doc 2 | source=faq | score=0.668]

Q: How do I get access to the project data stored in S3?
Type: generated
A: Install the VPN client, register your device in the IT portal, then use your company credentials.



In [9]:
# %%
# Simple CLI loop (runs in notebook too)
def chat_loop():
    print("Internship Support Chatbot — type 'exit' to quit")
    while True:
        q = input("\nYou: ").strip()
        if q.lower() in ("exit", "quit"):
            print("Bye!")
            break
        resp = generate_answer(q, top_k=3, use_generator=USE_GENERATOR)
        print("\nAssistant:", resp["answer"])

# Uncomment to run interactive loop (works in terminal)
# chat_loop()